In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import warnings
from IPython.display import display
current_dir = os.getcwd()
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root added to Python path: {project_root}")

Project root added to Python path: c:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\ipai_project


# **Check DWH connection**

In [2]:
load_dotenv(find_dotenv())

PROJECT_ROOT = os.getenv("PROJECT_ROOT")
IPAI_PROJECT_DIR = os.getenv("IPAI_PROJECT_DIR") # <-- Читаем новую директорию

if not IPAI_PROJECT_DIR:
    raise ValueError("IPAI_PROJECT_DIR is not set in the .env file!")

if IPAI_PROJECT_DIR not in sys.path:
    sys.path.append(IPAI_PROJECT_DIR)

from src.database.connection_manager import db_manager

print("Testing connection to AWS RDS...")

try:
    with db_manager.get_dwh_connection() as conn:
        with conn.cursor() as cursor:
            
            cursor.execute("SELECT VERSION();")
            version = cursor.fetchone()
            
            print(f"Success! Connected to AWS RDS.")
            print(f"Database version: {version[0]}")
            
            cursor.execute("SHOW DATABASES;")
            databases = [db[0] for db in cursor.fetchall()]
            print(f"Available databases: {databases}")
            
except Exception as e:
    print(f"Test error: {e}")

Testing connection to AWS RDS...
Success! Connected to AWS RDS.
Database version: 8.4.8
Available databases: ['eliqsir_dwh', 'information_schema', 'mysql', 'performance_schema', 'sys']


# **Build drug graphs**

### Technical Specification: Advanced Quantum Drug Dataset

**1. General Description**
This dataset represents a strictly typed HDF5 repository optimized for training Geometric and Quantum Graph Neural Networks (GNNs). Each molecule is encoded as a fully connected graph endowed with 3D spatial coordinates and quantum-chemical descriptors.

* **Storage Format:** HDF5 (Gzip compressed for optimized I/O).
* **Sample Volume:** ~1.5 million compounds (sourced from ChEMBL via the internal `dim_drug` Data Warehouse table).
* **Granularity:** All-atom resolution, including explicitly modeled hydrogen atoms to preserve spatial and electrostatic fidelity.

**2. Data Schema (Tensor Structure)**
For each molecular entity (indexed by `drug_key`), the dataset provides five foundational tensor matrices:

* **A. Node Features (Atom-level matrix)**
    Describes the precise chemical identity and local environment of each atom:
    1.  **Atomic Number:** Periodic table integer.
    2.  **Degree:** Number of covalently bound adjacent heavy atoms.
    3.  **Formal Charge:** Integer electrical charge dictated by valence state.
    4.  **Hybridization:** Orbital hybridization state (e.g., SP, SP2, SP3, encoded as categorical integers).
    5.  **Aromaticity:** Binary flag indicating membership in a conjugated aromatic system.
    6.  **Mass:** Exact atomic mass.
    7.  **Is in Ring:** Binary topological flag.
    8.  **Implicit Valence:** Number of implicitly calculated valences/hydrogens.
    9.  **Chirality:** Stereochemical configuration center (R/S enantiomeric encoding).
    10. **Gasteiger Charge:** Partial quantum charge calculated iteratively based on orbital electronegativity equalization.

* **B. Edge Index (Topology)**
    Adjacency matrix represented in Coordinate List (COO) format, defining the topological skeleton of the molecule.

* **C. Edge Features (Bond-level attributes)**
    1.  **Bond Type:** Single, Double, Triple, or Aromatic (integer or one-hot encoded).
    2.  **Is Conjugated:** Binary flag indicating if the bond participates in a delocalized pi-electron system.
    3.  **Is in Ring:** Binary flag for bond presence within a ring structure.
    4.  **Stereo:** Bond stereochemistry configuration (E/Z/None).

* **D. Positions (3D Geometry)**
    3D spatial coordinates (x, y, z) for each atom in Angstroms, extracted post-energy minimization.

* **E. Global Context (Graph-level physicochemical profile)**
    1.  **LogP:** Lipophilicity coefficient, estimating membrane permeability (calculated via the Wildman-Crippen atom-based method).
    2.  **TPSA:** Topological Polar Surface Area, evaluating drug transport properties (calculated using 2D contributions of nitrogen and oxygen fragments).
    3.  **MolWt:** Exact molecular weight.

**3. Data Provenance & Methodology**
* **Raw Data Source:** Canonical SMILES strings retrieved from the `dim_drug` table.
* **Topology Processing:** RDKit Sanitization module applied to clean structures, perceive aromaticity, and enforce strict valence rules.
* **3D Conformation Generation:** Computed via the ETKDG v3 (Experimental-Torsion-Knowledge Distance Geometry) algorithm to sample physically realistic spatial embeddings.
* **Geometry Optimization:** Minimized using the Universal Force Field (UFF) or MMFF94 to refine bond lengths and angles to local energy minima.
* **Quantum Features:** Derived using the Gasteiger-Marsili empirical algorithm to map realistic electrostatic charge distributions across the graph.

In [2]:
from src.database.connection_manager import ConnectionManager
from src.retrieval.micro_layer.drugs.engine import QuantumGraphEngine
from src.retrieval.micro_layer.hdf5_handler import HDF5Handler
from src.retrieval.micro_layer.drugs.pipeline import DrugPipeline

db_manager = ConnectionManager()

engine = QuantumGraphEngine()
main_storage = HDF5Handler(filename="advanced_quantum_dataset.h5") 
pipeline = DrugPipeline(db_manager=db_manager, engine=engine, storage=main_storage)

print("Launching FULL dataset generation...")
print("Note: This process will take a significant(!!!) amount of time. You can monitor the progress bar.")
pipeline.run(batch_size=1000)

QuantumGraphEngine initialized: Strict 10-feature mode active.
HDF5 Storage initialized. Target file: c:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\datasets\advanced_quantum_dataset.h5
Launching FULL dataset generation...
Note: This process will take a significant(!!!) amount of time. You can monitor the progress bar.
Starting Drug Graph Generation Pipeline...


Batch (offset 0):  33%|███▎      | 329/1000 [00:49<00:59, 11.23it/s][22:55:20] UFFTYPER: Unrecognized charge state for atom: 5
[22:55:20] UFFTYPER: Unrecognized charge state for atom: 5
Batch (offset 0):  39%|███▉      | 394/1000 [00:54<00:33, 18.35it/s][22:55:25] UFFTYPER: Unrecognized charge state for atom: 1
[22:55:25] UFFTYPER: Unrecognized charge state for atom: 1
Batch (offset 0):  51%|█████     | 512/1000 [01:10<03:12,  2.53it/s][22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
Batch (offset 0):  52%|█████▏    | 520/1000 [01:10<00:48,  9.88it/s][22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
Batch (offset 0):  52%|█████▏    | 523/1000 [01:10<00:36, 12.95it/s][22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] 


Critical Error saving batch to HDF5: [Errno 0] Unable to synchronously open file (unable to lock file, errno = 0, error message = 'No error', Win32 GetLastError() = 33)

PIPELINE COMPLETE!
Time elapsed: 46966.94 seconds
Successfully saved: 248413 graphs
Skipped (invalid SMILES or failed 3D/charges): 2591


In [5]:
from src.database.connection_manager import ConnectionManager
from src.retrieval.micro_layer.drugs.engine import QuantumGraphEngine
from src.retrieval.micro_layer.hdf5_handler import HDF5Handler
from src.retrieval.micro_layer.drugs.pipeline import DrugPipeline

db_manager = ConnectionManager()

# Initialize Engine & Storage
# Important: HDF5Handler will automatically open the existing file in append ('a') mode
engine = QuantumGraphEngine()
main_storage = HDF5Handler(filename="advanced_quantum_dataset.h5") 

# Initialize Pipeline
pipeline = DrugPipeline(db_manager=db_manager, engine=engine, storage=main_storage)
# Run the pipeline starting from the resume_offset
# Note: Ensure your pipeline.run() method accepts the 'offset' parameter 
# and passes it to the SQL query (e.g., LIMIT batch_size OFFSET offset)
pipeline.run(batch_size=1000, start_offset=1427000)
print(f"Resuming FULL dataset generation from offset {start_offset}...")


QuantumGraphEngine initialized: Strict 10-feature mode active.
HDF5 Storage initialized. Target file: c:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\datasets\advanced_quantum_dataset.h5
Starting Drug Graph Generation Pipeline from offset 1427000...


Batch (offset 1428000):   8%|▊         | 77/1000 [00:14<02:02,  7.53it/s][13:56:52] UFFTYPER: Unrecognized charge state for atom: 1
[13:56:52] UFFTYPER: Unrecognized charge state for atom: 1
Batch (offset 1428000):  14%|█▍        | 139/1000 [00:23<01:58,  7.29it/s][13:57:00] UFFTYPER: Unrecognized charge state for atom: 10
[13:57:00] UFFTYPER: Unrecognized charge state for atom: 10
Batch (offset 1428000):  16%|█▌        | 157/1000 [00:25<01:15, 11.13it/s][13:57:02] UFFTYPER: Unrecognized charge state for atom: 1
[13:57:02] UFFTYPER: Unrecognized charge state for atom: 1
Batch (offset 1428000):  30%|██▉       | 296/1000 [00:48<02:34,  4.57it/s][13:57:26] UFFTYPER: Unrecognized charge state for atom: 35
[13:57:26] UFFTYPER: Unrecognized charge state for atom: 35
Batch (offset 1428000):  99%|█████████▉| 988/1000 [03:04<00:03,  3.64it/s][13:59:41] UFFTYPER: Unrecognized charge state for atom: 21
[13:59:42] UFFTYPER: Unrecognized charge state for atom: 21
Batch (offset 1429000):  30%|███   

Reached the end of the database table.

PIPELINE COMPLETE!
Time elapsed: 22999.62 seconds
Successfully saved in this run: 82013 graphs
Skipped in this run: 919


NameError: name 'start_offset' is not defined